# Step 6: Temporal-Aware ALS 

## Phase 4: Hybrid Personalization

**Goal:** Train collaborative filtering model with temporal decay using pure Python

**Temporal-Aware Confidence:**
```
confidence(u,c) = 1 + alpha · R(u,c) · temporal_decay(t)
temporal_decay(t) = exp(-days_since_enroll / 365)
```

**ALS Algorithm:**
- Alternating Least Squares for matrix factorization
- User embeddings: U
- Course embeddings: V
- Rating matrix: R ≈ U · V^T

**Output:**
- als_embeddings.pkl: user_embeddings, course_embeddings
- als_recommendations.json: top-k recommendations

In [3]:
import json
import pickle
import numpy as np
import pandas as pd
from collections import defaultdict
from datetime import datetime
import math

OUTPUT_PATH = "C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data/output"

# Parameters
RANK = 100
ALPHA = 40
MAX_ITER = 12
REG_PARAM = 0.1
REFERENCE_DATE = datetime(2021, 1, 1)  # Reference for temporal decay

print("Step 6: Temporal-Aware ALS")
print(f"Parameters: rank={RANK}, alpha={ALPHA}, max_iter={MAX_ITER}, reg={REG_PARAM}")

Step 6: Temporal-Aware ALS
Parameters: rank=100, alpha=40, max_iter=12, reg=0.1


## Load Data

In [4]:
# Load user sequences
with open(f"{OUTPUT_PATH}/user_sequences.json", "r", encoding='utf-8') as f:
    user_sequences = json.load(f)

print(f"Loaded {len(user_sequences)} user sequences")

# Parse into structured format
records = []
for record in user_sequences:
    user_id = record["user_id"]
    sequence = record["sequence"]
    
    for item in sequence:
        sequence_order = item[0]
        course_id = item[1]
        r_score = item[2]
        passed = item[3]
        
        records.append({
            'user_id': user_id,
            'course_id': course_id,
            'sequence_order': sequence_order,
            'R_score': r_score,
            'passed': passed
        })

# Create DataFrame
df = pd.DataFrame(records)
print(f"Total records: {len(df):,}")
print(f"Unique users: {df['user_id'].nunique():,}")
print(f"Unique courses: {df['course_id'].nunique():,}")
df.head()

Loaded 11385 user sequences
Total records: 71,479
Unique users: 11,385
Unique courses: 744


,user_id,course_id,sequence_order,R_score,passed
0,U_173698,C_696908,1,0.328,0
1,U_173698,C_1756060,2,0.328,0
2,U_25846,C_682344,3,0.398,0
3,U_25846,C_676953,14,0.398,0
4,U_378400,C_770784,4,0.307,0


## Calculate Temporal Decay

In [6]:

def temporal_decay(sequence_order, max_order):
    """Calculate temporal decay based on sequence order"""
    normalized = sequence_order / max_order if max_order > 0 else 0.5
    decay = math.exp(-(1 - normalized))
    return max(decay, 0.2)


def compute_r_norm(df):
    user_max_r = df.groupby('user_id')['R_score'].transform('max')
    df['R_norm'] = np.where(user_max_r > 0, df['R_score'] / user_max_r, 0)
    return df


def compute_temporal_decay(df):
    user_max_order = df.groupby('user_id')['sequence_order'].transform('max')

    normalized = np.where(
        user_max_order > 0,
        df['sequence_order'] / user_max_order,
        0.5
    )

    decay = np.exp(-(1 - normalized))
    df['temporal_decay'] = np.maximum(decay, 0.2)
    return df


def compute_confidence(df, ALPHA):
    df['confidence'] = 1 + ALPHA * df['R_norm'] * df['temporal_decay']
    df['confidence'] = (df['confidence'] - df['confidence'].min()) / (df['confidence'].max() - df['confidence'].min())
    return df


# pipeline
df = compute_r_norm(df)
df = compute_temporal_decay(df)
df = compute_confidence(df, ALPHA)

print("Temporal decay applied")
print(df[['user_id', 'course_id', 'R_norm', 'temporal_decay', 'confidence']].head())

Temporal decay applied
    user_id  course_id  R_norm  temporal_decay  confidence
0  U_173698   C_696908     1.0        0.606531    0.377239
1  U_173698  C_1756060     1.0        1.000000    1.000000
2   U_25846   C_682344     1.0        0.455794    0.138661
3   U_25846   C_676953     1.0        1.000000    1.000000
4  U_378400   C_770784     1.0        0.548812    0.285884


## Create ID Mappings

In [7]:
# Create ID to index mappings
unique_users = df['user_id'].unique()
unique_courses = df['course_id'].unique()

user_to_idx = {user: idx for idx, user in enumerate(unique_users)}
course_to_idx = {course: idx for idx, course in enumerate(unique_courses)}

idx_to_user = {idx: user for user, idx in user_to_idx.items()}
idx_to_course = {idx: course for course, idx in course_to_idx.items()}

n_users = len(unique_users)
n_courses = len(unique_courses)

print(f"Users: {n_users:,}")
print(f"Courses: {n_courses:,}")
print(f"Density: {len(df) / (n_users * n_courses):.4%}")

Users: 11,385
Courses: 744
Density: 0.8439%


## Build Rating Matrix

In [8]:
# Build sparse rating matrix
# Store as dictionary for efficiency
ratings = {}  # (user_idx, course_idx) -> confidence

for _, row in df.iterrows():
    user_idx = user_to_idx[row['user_id']]
    course_idx = course_to_idx[row['course_id']]
    ratings[(user_idx, course_idx)] = row['confidence']

# Group by user
user_ratings = defaultdict(dict)
for (u, c), rating in ratings.items():
    user_ratings[u][c] = rating

# Group by course
course_ratings = defaultdict(dict)
for (u, c), rating in ratings.items():
    course_ratings[c][u] = rating

print(f"Rating entries: {len(ratings):,}")
print(f"Avg ratings per user: {len(ratings) / n_users:.2f}")
print(f"Avg ratings per course: {len(ratings) / n_courses:.2f}")

Rating entries: 71,479
Avg ratings per user: 6.28
Avg ratings per course: 96.07


## ALS Algorithm

In [9]:
def initialize_embeddings(n_users, n_items, rank):
    """Initialize user and item embeddings with small random values"""
    np.random.seed(42)
    user_emb = np.random.normal(0, 0.01, (n_users, rank))
    item_emb = np.random.normal(0, 0.01, (n_items, rank))
    # Make positive
    user_emb = np.abs(user_emb)
    item_emb = np.abs(item_emb)
    return user_emb, item_emb

def update_user_embedding(user_idx, user_emb, item_emb, user_ratings, reg_param, rank):
    """Update single user embedding using least squares"""
    rated_items = user_ratings[user_idx]
    
    if len(rated_items) == 0:
        return user_emb[user_idx]
    
    # Build matrix A (item embeddings) and vector b (ratings)
    n_ratings = len(rated_items)
    A = np.zeros((n_ratings, rank))
    b = np.zeros(n_ratings)
    
    for i, (item_idx, rating) in enumerate(rated_items.items()):
        A[i] = item_emb[item_idx]
        b[i] = rating
    
    # Add regularization
    A_reg = np.vstack([A, np.sqrt(reg_param) * np.eye(rank)])
    b_reg = np.hstack([b, np.zeros(rank)])
    
    # Solve least squares
    try:
        solution, _, _, _ = np.linalg.lstsq(A_reg, b_reg, rcond=None)
        return np.maximum(solution, 0)  # Non-negative
    except:
        return user_emb[user_idx]

def update_item_embedding(item_idx, user_emb, item_emb, item_ratings, reg_param, rank):
    """Update single item embedding using least squares"""
    rated_users = item_ratings[item_idx]
    
    if len(rated_users) == 0:
        return item_emb[item_idx]
    
    n_ratings = len(rated_users)
    A = np.zeros((n_ratings, rank))
    b = np.zeros(n_ratings)
    
    for i, (user_idx, rating) in enumerate(rated_users.items()):
        A[i] = user_emb[user_idx]
        b[i] = rating
    
    A_reg = np.vstack([A, np.sqrt(reg_param) * np.eye(rank)])
    b_reg = np.hstack([b, np.zeros(rank)])
    
    try:
        solution, _, _, _ = np.linalg.lstsq(A_reg, b_reg, rcond=None)
        return np.maximum(solution, 0)
    except:
        return item_emb[item_idx]

def train_als(user_ratings, course_ratings, n_users, n_courses, rank, max_iter, reg_param):
    """Train ALS model"""
    print(f"Training ALS: rank={rank}, max_iter={max_iter}")
    
    # Initialize
    user_emb, item_emb = initialize_embeddings(n_users, n_courses, rank)
    
    # Training loop
    for iteration in range(max_iter):
        # Update user embeddings
        for u in range(n_users):
            user_emb[u] = update_user_embedding(u, user_emb, item_emb, user_ratings, reg_param, rank)
        
        # Update item embeddings
        for c in range(n_courses):
            item_emb[c] = update_item_embedding(c, user_emb, item_emb, course_ratings, reg_param, rank)
        
        # Calculate loss every few iterations
        if (iteration + 1) % 5 == 0:
            loss = 0
            count = 0
            for (u, c), rating in ratings.items():
                pred = np.dot(user_emb[u], item_emb[c])
                loss += (rating - pred) ** 2
                count += 1
            # Add regularization
            loss += reg_param * (np.sum(user_emb ** 2) + np.sum(item_emb ** 2))
            rmse = np.sqrt(loss / count) if count > 0 else 0
            print(f"  Iteration {iteration + 1}/{max_iter}, RMSE: {rmse:.4f}")
    
    return user_emb, item_emb

# Train model
print("Starting ALS training...")
user_embeddings, course_embeddings = train_als(
    user_ratings,
    course_ratings,
    n_users,
    n_courses,
    RANK,
    MAX_ITER,
    REG_PARAM
)

print(f"\nTraining complete!")
print(f"User embeddings shape: {user_embeddings.shape}")
print(f"Course embeddings shape: {course_embeddings.shape}")

Starting ALS training...
Training ALS: rank=100, max_iter=12
  Iteration 5/12, RMSE: 0.2711
  Iteration 10/12, RMSE: 0.2850

Training complete!
User embeddings shape: (11385, 100)
Course embeddings shape: (744, 100)


## Map Embeddings Back to IDs

In [10]:
# Map back to original IDs
user_embeddings_mapped = {
    idx_to_user[idx]: emb.tolist()
    for idx, emb in enumerate(user_embeddings)
}

course_embeddings_mapped = {
    idx_to_course[idx]: emb.tolist()
    for idx, emb in enumerate(course_embeddings)
}

print(f"Mapped user embeddings: {len(user_embeddings_mapped)}")
print(f"Mapped course embeddings: {len(course_embeddings_mapped)}")
print(f"Embedding dimension: {RANK}")

Mapped user embeddings: 11385
Mapped course embeddings: 744
Embedding dimension: 100


## Generate Recommendations

In [11]:
def generate_recommendations(user_idx, user_emb, item_emb, user_ratings, top_k=10):
    """Generate top-k recommendations for a user"""
    # Get courses already rated
    rated_courses = set(user_ratings[user_idx].keys())
    
    # Calculate scores for all courses
    scores = np.dot(item_emb, user_emb[user_idx])
    
    # Mask already rated courses
    for c in rated_courses:
        scores[c] = -np.inf
    
    # Get top-k
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    return [(idx, float(scores[idx])) for idx in top_indices if scores[idx] > -np.inf]

# Generate recommendations for all users
print("Generating recommendations...")
recommendations = {}

for user_idx in range(n_users):
    user_id = idx_to_user[user_idx]
    recs = generate_recommendations(user_idx, user_embeddings, course_embeddings, user_ratings, top_k=10)
    
    recommendations[user_id] = [
        {
            "course_id": idx_to_course[course_idx],
            "als_score": score
        }
        for course_idx, score in recs
    ]

print(f"Generated recommendations for {len(recommendations)} users")

# Show example
example_user = list(recommendations.keys())[0]
print(f"\nExample recommendations for {example_user}:")
for rec in recommendations[example_user][:5]:
    print(f"  {rec['course_id']}: {rec['als_score']:.4f}")

Generating recommendations...
Generated recommendations for 11385 users

Example recommendations for U_173698:
  C_697821: 1.3733
  C_696841: 1.2623
  C_677236: 1.2595
  C_677049: 1.2413
  C_697791: 1.1420


## Save Results

In [12]:
# Save embeddings
embeddings_data = {
    "user_embeddings": user_embeddings_mapped,
    "course_embeddings": course_embeddings_mapped,
    "als_params": {
        "rank": RANK,
        "alpha": ALPHA,
        "max_iter": MAX_ITER,
        "reg_param": REG_PARAM
    },
    "user_to_idx": user_to_idx,
    "course_to_idx": course_to_idx
}

with open(f"{OUTPUT_PATH}/als_embeddings.pkl", "wb") as f:
    pickle.dump(embeddings_data, f)

# Save recommendations
with open(f"{OUTPUT_PATH}/als_recommendations.json", "w", encoding='utf-8') as f:
    json.dump(recommendations, f, indent=2)

print(f"\nSaved:")
print(f"  - als_embeddings.pkl")
print(f"  - als_recommendations.json")


Saved:
  - als_embeddings.pkl
  - als_recommendations.json


## Summary

In [13]:
print("="*60)
print("STEP 6 COMPLETE!")
print("="*60)
print(f"\nALS Model:")
print(f"  - Rank: {RANK}")
print(f"  - Users: {len(user_embeddings_mapped):,}")
print(f"  - Courses: {len(course_embeddings_mapped):,}")
print(f"  - Recommendations: {len(recommendations):,} users")
print(f"\nNext: Step 7 - Hybrid Recommendation & Evaluation")

STEP 6 COMPLETE!

ALS Model:
  - Rank: 100
  - Users: 11,385
  - Courses: 744
  - Recommendations: 11,385 users

Next: Step 7 - Hybrid Recommendation & Evaluation
